In [ ]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers

# Some other useful packages 
import importlib
from pathlib import Path


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )

Rdair=Co.Rdair()


In [ ]:
# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



In [ ]:
%%time
nsteps=None
start_date=None
#super_lat_range = [-90.,90.]  #[-85,-30]
super_lat_range = [-80.,-30.]  #[-85,-30]
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x02'   , False, [2004,7,15,0], 248
case, process_ncdata  = 'cam77_dyamond1_prod1'    , False
#case , process_ncdata = 'xy-rdg-mm-front'    , True
A = futi.read_case( case=case, nsteps=nsteps, start_date=start_date , super_lat_range=super_lat_range ) # , nsteps = 31*8 )

time, zlev, lat, lon = A.time, A.zlev, A.lat, A.lon



In [ ]:
#################################################################
# Make event lists ... and composites
importlib.reload(euti)
importlib.reload(auti)

thresh=0.01 #0.005 # 0.02
thresh=0.001 #0.005 # 0.02
thresh=0.0001 #0.005 # 0.02
zlev_event=10_000. #23_000.
zlev_event=15_000. #23_000.

El=[]

print( f"This run use dycore={A.dycore}")

if A.dycore == 'MPAS':
    ## for MPAS 3km
    thresholds=[  [0.0,1e6],
                [0.005,1e6], 
                [0.008,1e6],
                [0.01,1e6] ,
                [0.020,1e6]  ] ##[0.025,1e6]  ] ##0.025, ]
    second_thresholds= [
                    [0.005,0.008], 
                    [0.008,0.015],
                    [0.01,0.025] ,
                    [0.020,1e6]  ] ## [0.025,1e6]  ] ##0.025, ]
    big_window=[3,5,5]
    lil_window=[3,2,2]
    
elif A.dycore == 'SE':
    # for ne240
    thresholds=[  [0.0005,1e6], 
                [0.002,1e6],
                [0.005,1e6] ,
                [0.010,1e6]  ] ##[0.025,1e6]  ] ##0.025, ]
    second_thresholds= [
                    [0.0005,0.002], 
                    [0.002,0.0075],
                    [0.005,0.012] ,
                    [0.012,1e6]  ] ## [0.025,1e6]  ] ##0.025, ]
    big_window=[2,5,5]
    lil_window=[2,2,2]

lat_range=  [-65,-40] #[-70,-60] #[-60,-40]
lon_range=[0,60] # [0,60]
lat_range=  [-50,-40] #[-70,-60] #[-60,-40]
#lat_range=  [-60,-50] #[-70,-60] #[-60,-40]
lon_range=[0,360] # [0,60]

ithr=0
for thresh in thresholds:
    second_thresh=None #second_thresholds[ithr]
    ds =euti.make_ds(fld=A.rho_epwp[:,:,:,:], lon=lon, lat=lat, zlev=zlev, time=time, 
                     thresh=thresh,second_thresh=second_thresh,zlev_event=zlev_event, 
                     lat_range=lat_range, lon_range=lon_range)
    ithr=ithr+1
    

    # get shape of varaiables
    nt,nz,ny,nx = np.shape( A.u )
    
    htopo_t = np.tile(A.htopo[None, :, :], ( nt, 1, 1))
    htopo_4D_x , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=htopo_t , lon=lon, lat=lat, window=[0,5,5] , TZHkey='tyx', lat_range=[-999,999], lon_range=[-999,999] )
    
    htopo_MMM = auti.collapseSpace( htopo_4D_x , TZHkey='etyx')
    htopo_super_max = htopo_MMM[2].max( axis=1 )
    
    
    ########################################
    # Exclude events with topography nearby
    ########################################
    
    flat=np.where(htopo_super_max<0.0001)
    flat[0].shape
    
    ds_flat=ds.isel( index=flat[0] )    
    ds=ds_flat
    
    E_ = {'ds':ds }
    E = AttrDict( E_ )
    El.append(E)



In [ ]:
plt.plot( El[0].ds.epwp_max)

In [ ]:
x=El[0].ds.epwp_max.values
print( x.min() , x.max() )

In [ ]:
poo,xcen=auti.one_dim_pdf( x )

In [ ]:
importlib.reload(auti)
cumu,xs=auti.cumul_big_to_small( x, plot_it=True )

In [ ]:
boo=[0.99,0.9,.5,.25,.125,.05]
for b in boo:
    xoo=np.argmin( np.abs(cumu-b) )
    print(f" fraction={100*b:4.1f}% of total epwp is in events with epwp > {xs[xoo]:.5f}. Carried by N={len(xs[xoo:]):6d} events or {100*len(xs[xoo:])/len(xs):5.2f}%  ")

In [ ]:
plt.plot( xcen, poo)